# Backtest analysis

This notebook reads saved outputs only. It does not train models, execute trades, or run backtests.
The currently available results are **15 synthetic accounting scenarios**, which provide no evidence of strategy profitability.
The historical analysis section will be connected after the 2021–2022 backtest is complete; results from 2023 onward and OOS data are not loaded.

Nautilus provides order, fill, position, and account reports, as well as tearsheets.
This notebook adds our research metrics, benchmark comparisons, and export formats.
[Official reporting documentation](https://nautilustrader.io/docs/latest/concepts/reports/)


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/momentum_data/backtest_analysis.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from this repository")
sys.path.insert(0, str(ROOT / "src"))
from momentum_data.backtest_analysis import (  # noqa: E402 - repository source path
    load_development_run,
    plot_development,
    plot_smoke,
    smoke_tables,
)

SMOKE_REPORT = ROOT / "data/interim/nautilus_smoke/v20261120_001/summary.json"
# Change only after a real 2021–2022 run exports the documented report contract.
# Example: ROOT / "data/interim/backtests/development_run_001"
HISTORICAL_RUN = None
EXPORT = True
FIGURES = ROOT / "reports/figures/backtest_analysis"
TABLES = ROOT / "reports/tables/backtest_analysis"
if EXPORT:
    FIGURES.mkdir(parents=True, exist_ok=True)
    TABLES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})
print("Repository:", ROOT)
print("Historical performance enabled:", HISTORICAL_RUN is not None)

## 1. Synthetic accounting checks

Each row is an independent scenario, not a continuous portfolio. **Do not concatenate these scenarios into an equity curve or calculate Sharpe across them.**
Check fills, entry and exit commissions, funding, and slippage against hand-calculated values.

In [ ]:
smoke, fills, smoke_meta = smoke_tables(SMOKE_REPORT)
display(pd.Series({k: smoke_meta[k] for k in [
    "success", "scenarios", "repeat_runs_identical", "nautilus_trader",
    "real_market_strategy_performance_computed", "OOS_performance_inspected",
]}, name="Accounting run metadata"))
display(smoke.round(8))
display(fills.head(10))
fig = plot_smoke(smoke)
if EXPORT:
    smoke.to_csv(TABLES / "synthetic_scenarios.csv")
    fills.to_csv(TABLES / "synthetic_fills.csv", index=False)
    fig.savefig(FIGURES / "synthetic_accounting.png", dpi=180)
plt.show()

## 2. Historical performance: equity, drawdown, monthly returns, exposure, and costs

Data is loaded only when `HISTORICAL_RUN` is set. Inputs must explicitly identify the 2021–2022 development period
and include strategy and BTC/ETH benchmark NAV on the same dates, valued in USDT.
Strategy NAV must include unrealized PnL, commissions, and funding; account balance alone is not portfolio equity.

Metrics: total return, CAGR, annualized volatility, Sharpe, MDD, and Calmar. Crypto returns use 365-day annualization,
the sample standard deviation of daily returns, and a zero risk-free rate. Sharpe is undefined for zero volatility; Calmar is undefined for zero drawdown.
Annualized values from short samples are diagnostic only; the first and last calendar months may be incomplete.
In cost charts, commissions are positive expenses and funding is positive for income and negative for payments.
Turnover is total daily traded notional divided by previous-day NAV, including both entry and exit trades, with no division by two.

Missing days, duplicate dates, NaN values, and nonpositive NAV raise errors rather than being hidden by forward filling.

In [ ]:
daily = manifest = metrics = monthly = None
if HISTORICAL_RUN is None:
    print("No completed historical strategy output is available. Equity, Sharpe, and benchmark performance are not generated.")
else:
    daily, manifest = load_development_run(Path(HISTORICAL_RUN))
    display(pd.Series(manifest, name="Historical run manifest"))
    fig, metrics, monthly = plot_development(daily)
    display(metrics)
    display(monthly)
    if EXPORT:
        fig.savefig(FIGURES / "development_dashboard.png", dpi=180)
        metrics.to_csv(TABLES / "development_metrics.csv")
        monthly.to_csv(TABLES / "development_monthly_returns.csv")
        daily.to_csv(TABLES / "development_daily.csv")
    plt.show()

## 3. Model interpretation and experiment comparisons (pending integration)

SHAP is required in the formal model report. No model has been fitted yet, so no SHAP values are generated or fabricated.
Future inputs will include SHAP for each validation fold, factor ablations, 1/2/4-week rebalancing comparisons, and buffer experiments.
Each run must retain its own configuration and results; reporting only the best run is insufficient.

This notebook does not unseal 2023, 2024, or the final OOS period. Changes to the evaluation stage require separate registration.
See `docs/BACKTEST_NOTEBOOK.md` for the complete input schema and launch instructions.

In [ ]:
if EXPORT:
    provenance = {
        "notebook": "notebooks/backtest_analysis.ipynb",
        "synthetic_report": str(SMOKE_REPORT),
        "synthetic_cases_sha256": smoke_meta["cases_sha256"],
        "historical_run": str(HISTORICAL_RUN) if HISTORICAL_RUN else None,
        "OOS_performance_inspected": False,
        "note": "Synthetic scenarios are separate fixtures, not a portfolio return series.",
    }
    (TABLES / "analysis_provenance.json").write_text(
        json.dumps(provenance, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )
    print("Figures:", FIGURES)
    print("Tables:", TABLES)